# 📊 Student Engagement & Outcomes Tracker
**Author:** Sailee Choudhari  
**Description:** End-to-end analysis of student engagement, persistence, and outcomes across 5 semesters and 4 experiential learning programs. Combines quantitative pre/post survey analysis, multi-semester trend synthesis, qualitative theme extraction, and equity analysis.

---

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from IPython.display import display

from analysis import (
    load_and_clean, demographic_summary, pre_post_summary,
    persistence_outcomes, semester_trends, equity_analysis,
    extract_themes, sentiment_split, generate_narrative
)
from visualizations import generate_all_charts

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)
print('Imports OK')

## 1. Load & Clean Data

In [ ]:
df = load_and_clean('../data/student_engagement_data.csv')
print(f'Records: {len(df)} | Columns: {len(df.columns)}')
print(f'Semesters: {sorted(df["semester"].unique())}')
print(f'Programs: {df["program"].unique().tolist()}')
df.head()

## 2. Demographic Breakdown

In [ ]:
demo = demographic_summary(df)
for col, table in demo.items():
    print(f'\n── {col.upper()} ──')
    display(table)

## 3. Pre/Post Survey Analysis

In [ ]:
print('=== OVERALL ===')
display(pre_post_summary(df))

print('\n=== BY PROGRAM ===')
display(pre_post_summary(df, groupby='program'))

print('\n=== BY SEMESTER ===')
display(pre_post_summary(df, groupby='semester'))

## 4. Persistence & Outcomes by Program

In [ ]:
outcomes = persistence_outcomes(df)
display(outcomes)

print(f'\nBest persistence: {outcomes["persistence_rate"].idxmax()} ({outcomes["persistence_rate"].max()}%)')
print(f'Best internship rate: {outcomes["internship_rate"].idxmax()} ({outcomes["internship_rate"].max()}%)')

## 5. Multi-Semester Trend Analysis

In [ ]:
trends = semester_trends(df)
display(trends)

# Flag semesters with declining engagement
gain_series = trends['avg_overall_gain']
drops = gain_series[gain_series.diff() < 0]
if len(drops):
    print(f'\n⚠ Semesters with declining gains: {drops.index.tolist()}')
else:
    print('\n✓ No declining gain semesters detected')

## 6. Equity Analysis

In [ ]:
equity = equity_analysis(df)
display(equity)

fg = df[df['first_generation']==True]['overall_gain'].mean()
nfg = df[df['first_generation']==False]['overall_gain'].mean()
print(f'\nFirst-gen avg gain: {fg:.3f}  |  Non-first-gen avg gain: {nfg:.3f}  |  Gap: {nfg-fg:+.3f}')

## 7. Qualitative Theme Extraction (Mixed-Methods)

In [ ]:
themes = extract_themes(df)
sentiment = sentiment_split(df)

print('── THEMES ──')
display(themes)
print(f'\n── SENTIMENT ──')
for k,v in sentiment.items():
    print(f'  {k}: {v}')

print('\n── SAMPLE RESPONSES BY THEME ──')
print('Positive:', df[df['post_engagement']>=4]['open_response'].iloc[0])
print('Improvement:', df[df['post_engagement']<=2]['open_response'].iloc[0])

## 8. Visualizations

In [ ]:
chart_paths = generate_all_charts(df, themes, outdir='../reports/figures')

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for ax, path in zip(axes.flat, chart_paths):
    img = mpimg.imread(path)
    ax.imshow(img); ax.axis('off')
plt.suptitle('Student Engagement & Outcomes — Dashboard', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 9. Narrative Summary Report

In [ ]:
report = generate_narrative(df)
print(report)

with open('../reports/summary_report.txt', 'w') as f:
    f.write(report)
print('\n✓ Report saved to reports/summary_report.txt')